In [1]:
import pandas as pd
import numpy as np

SEED = 42
B = 5000

CRITICAL = {90: 1.645, 95: 1.96, 99: 2.576}

In ./data/ames_prices.csv, create an indicator that equals 1 when Central.Air == 'Y' and
0 otherwise.

a. Estimate the proportion of homes with central air.

In [ ]:
df = pd.read_csv("./data/ames_prices.csv")

df["central_air_indicator"] = (
    df["Central.Air"] == "Y"
).astype(int)

proportion = df["central_air_indicator"].mean()

print("proportion: ", proportion)

0.9331058020477816


Approximately 93.31% of homes have central air.

b. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this
proportion. Report the seed and number of bootstrap replications.

In [11]:
seed = 42
B = 5000

rng = np.random.default_rng(seed)

data = df["central_air_indicator"].to_numpy()

bootstrap_estimates = []

for i in range(B):
    sample = rng.choice(
        data,
        size=len(data),
        replace=True
    )

    bootstrap_estimates.append(sample.mean())

The seed is 42 and the number of bootstraps is 5,000. 

#Each bootstrap sample has 2,390 observations.

c. Compute the bootstrap standard error as the sample standard deviation of the bootstrap
estimates (using ddof=1 ensures that there's the 1/(N− 1)
degrees of freedom correction).

In [13]:
bootstrap_se = np.std(
    bootstrap_estimates,
    ddof=1
)

print("Standard Error of boostraps: ", bootstrap_se)

Standard Error of boostraps:  0.004692462621398659


d. Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and
99% confidence intervals.

In [14]:
critical_values = {
    90: 1.645,
    95: 1.96,
    99: 2.576
}

intervals = {}

for level, critical in critical_values.items():
    lower = proportion - critical * bootstrap_se
    upper = proportion + critical * bootstrap_se

    intervals[level] = (lower, upper)

    print(f"{level}% CI: ({lower:.4f}, {upper:.4f})")

90% CI: (0.9254, 0.9408)
95% CI: (0.9239, 0.9423)
99% CI: (0.9210, 0.9452)


e. Use each confidence interval to evaluate H :
0 p=
.90 α =
. At the matching significance levels
.10, .05, .01
, state whether you reject or fail to reject the null hypothesis. Briefly interpret your
results.

In [15]:
null_value = 0.90

for level, (lower, upper) in intervals.items():
    alpha = 1 - level / 100

    if lower <= null_value <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"alpha = {alpha:.2f}: {decision}")

alpha = 0.10: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


Interpretation: 

This sample shows that 93.31% of homes have central air, not 90% as stated in our null hypothesis. 

We reject the null hypothesis at all significance levels since 0.9 is outside each of our confidence intervals. So, this provides statistical evidence that the population proportion of homes with central air differs from 90%. Our estimated proportion was calculated to be 93.31%, this could meaan that the actual proportion can be higher than 90%.

In ./data/ames_prices.csv, estimate the Pearson correlation between np.arcsinh(area) and
np.arcsinh(price).

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the
correlation by resampling paired rows. Report the seed and number of bootstrap replications.

In [ ]:
df = pd.read_csv("./data/ames_prices.csv")

data = df[["area", "price"]].dropna()

area = np.arcsinh(data["area"])
price = np.arcsinh(data["price"])

correlation = area.corr(price)

print("Correlation: ", correlation)

Correlation:  0.7231661530091358


In [18]:
seed = 42
B = 5000

rng = np.random.default_rng(seed)

paired = np.column_stack((area, price))

bootstrap_correlations = []

for i in range(B):
    indices = rng.choice(
        len(paired),
        size=len(paired),
        replace=True
    )

    sample = paired[indices]

    r = np.corrcoef(
        sample[:, 0],
        sample[:, 1]
    )[0, 1]

    bootstrap_correlations.append(r)

- The original pearson correlation showed us a value of 0.7232
- The random seed is 42.
- The number of bootstrap replications is 5,000. 
- The method for sampling was resampling paired rows with replacement. 

b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap
estimates (using ddof=1 ensures that there's the 1/(N− 1)
degrees of freedom correction).

In [20]:
bootstrap_se = np.std(
    bootstrap_correlations,
    ddof=1
)

print("Standard Error of bootstrap: ", bootstrap_se)

Standard Error of bootstrap:  0.010223900961500762


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%,
95%, and 99% confidence intervals.

In [21]:
critical_values = {
    90: 1.645,
    95: 1.96,
    99: 2.576
}

intervals = {}

for level, z in critical_values.items():
    lower = correlation - z * bootstrap_se
    upper = correlation + z * bootstrap_se

    intervals[level] = (lower, upper)

    print(f"{level}% CI: ({lower:.4f}, {upper:.4f})")

90% CI: (0.7063, 0.7400)
95% CI: (0.7031, 0.7432)
99% CI: (0.6968, 0.7495)


d. Use each confidence interval to evaluate H0:
ρ= 0. At the matching significance levels α =
.10, .05, .01
, state whether you reject or fail to reject the null hypothesis. Briefly interpret the
association without making a causal claim.

In [22]:
null_value = 0

for level, (lower, upper) in intervals.items():
    alpha = 1 - level / 100

    if lower <= null_value <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"alpha = {alpha:.2f}: {decision}")

alpha = 0.10: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


Interpretation: Reject the null hypothesis at all three significance levels. 0 is outside all of the confidence intervals. This gives us statistical evidence of a positive relationship between the house area and the transformed price. It doesn't tell us that larger house area gives us higher prices though. We're only finding that there is a relationship between our variables.

e. Briefly explain why area and price must remain paired during resampling.

We have to keep the area connected to the corresponding price because we could accidentally use House A's area with a different house's price. So, this would change the relationship between the two variables and give us the wrong correlation to these variables. Each pair of these values belongs to the same house. If we do it separately, it would break the relationship that they have.

In ./data/metabric.csv, use Overall Survival (Months) to estimate the mean recorded
time to death or last follow-up. Include both living and deceased patients.

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the
mean. Report the seed and number of bootstrap replications.

In [25]:
df = pd.read_csv("./data/metabric.csv")

survival = df["Overall Survival (Months)"].dropna()

mean_survival = survival.mean()

print(mean_survival)

129.09429138744105


In [26]:
seed = 42
B = 5000

rng = np.random.default_rng(seed)

bootstrap_means = []

for i in range(B):
    sample = rng.choice(
        survival,
        size=len(survival),
        replace=True
    )

    bootstrap_means.append(sample.mean())

The random seed is 42.

The number of bootstrap replications is 5,000.

The original estimated mean was: 129.09 months.

We sampled with replacement. We had 1,343 observations per each bootstrap.

b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap
estimates (using ddof=1 ensures that there's the 1/(N− 1)
degrees of freedom correction).

In [27]:
bootstrap_se = np.std(
    bootstrap_means,
    ddof=1
)

print("Standard Error of bootstraps: ", bootstrap_se)

Standard Error of bootstraps:  2.142386022768364


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%,
95%, and 99% confidence intervals.

In [28]:
critical_values = {
    90: 1.645,
    95: 1.96,
    99: 2.576
}

intervals = {}

for level, z in critical_values.items():
    lower = mean_survival - z * bootstrap_se
    upper = mean_survival + z * bootstrap_se

    intervals[level] = (lower, upper)

    print(f"{level}% CI: ({lower:.2f}, {upper:.2f})")

90% CI: (125.57, 132.62)
95% CI: (124.90, 133.29)
99% CI: (123.58, 134.61)


d. Use each confidence interval to evaluate H :0 μ= 120
months. At the matching significance
levels α =
.10, .05, .01
, state whether you reject or fail to reject the null hypothesis.

In [29]:
null_value = 120

for level, (lower, upper) in intervals.items():
    alpha = 1 - level / 100

    if lower <= null_value <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"alpha = {alpha:.2f}: {decision}")

alpha = 0.10: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


We reject the null hypothesis at each of the intervals. The null hypothesis value we set as 120 is outside each of our intervals. This gives us statistical evidence that the popuylation mean recorded differs than the 120 value that we stated. 

e. Interpret the estimate and confidence intervals in both months and years. Because living patients'
eventual death times are right-censored, explain why these results do not estimate unrestricted
mean lifetime.

The estimated average survival time is 129.09 months. We also calculated these intervals:
1. 90% confidence interval, the interval contained [125.57, 132.62] months. We are 90% confident the population average recorded value falls within this range.
2. 95% confidence interval, the interval contained [124.90, 133.29] months. We are 95% confident that the population recorded value falls within this range.
3. 99% Confidence interval, the interval contained [123.58, 134.61] months. We are 99% confident that the population average value falls within this range. 

These results do not try to estimate the patient's average lifetime because some of the patients recorded were still alive at the last followup. We do not know their real value after the data was recorded, so this data is being right censored currently. The death times are unknown. The estimates as such do describe that last value recorded as the value of recorded time to death or the last follow-up. 

10. In ./data/nhanes_data_17_18.csv, select respondents who are at least 18 years old and have an
observed value for SleepHoursWeekdaysOrWorkdays. Estimate their mean weekday sleep.

a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean.
Report the seed and number of bootstrap replications.

In [31]:
df = pd.read_csv("./data/nhanes_data_17_18.csv", low_memory=False)

adults = df[df["AgeInYearsAtScreening"] >= 18]

sleep = adults["SleepHoursWeekdaysOrWorkdays"].dropna()

mean_sleep = sleep.mean()

print("Number of observations:", len(sleep))
print("Average sleep:", mean_sleep)

Number of observations: 5489
Average sleep: 7.62124248496994


In [32]:
seed = 42
B = 5000

rng = np.random.default_rng(seed)

bootstrap_means = []

for i in range(B):
    sample = rng.choice(
        sleep,
        size=len(sleep),
        replace=True
    )

    bootstrap_means.append(sample.mean())

- The original estimated mean was 7.6212 hours.
- The random seed is 42.
- The number of bootstrap replications is 5,000.
- The bootstrap samples were sampled with replacement. 

b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates
(ddof=1).

In [34]:
bootstrap_se = np.std(
    bootstrap_means,
    ddof=1
)

print("Boostrap standard error calculated: ", bootstrap_se)

Boostrap standard error calculated:  0.022587204784927532


c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

In [35]:
critical_values = {
    90: 1.645,
    95: 1.96,
    99: 2.576
}

intervals = {}

for level, z in critical_values.items():
    lower = mean_sleep - z * bootstrap_se
    upper = mean_sleep + z * bootstrap_se

    intervals[level] = (lower, upper)

    print(f"{level}% CI: ({lower:.4f}, {upper:.4f})")

90% CI: (7.5841, 7.6584)
95% CI: (7.5770, 7.6655)
99% CI: (7.5631, 7.6794)


d. Use each confidence interval to evaluate H : 0 μ= 8
hours. At the matching significance levels α =
.10, .05, .01
, state whether you reject or fail to reject the null hypothesis.

In [36]:
null_value = 8

for level, (lower, upper) in intervals.items():
    alpha = 1 - level / 100

    if lower <= null_value <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"alpha = {alpha:.2f}: {decision}")

alpha = 0.10: Reject H0
alpha = 0.05: Reject H0
alpha = 0.01: Reject H0


We reject the null hypothesis at each of the significance levels as the 8 hours stated in the null hypothesis is outside each of the confidence intervals. The data gives statistical evidence that the population mean sleep duration during the week differs from the 8 hours originally stated. 

e. Interpret the result and discuss whether its statistical and practical significance tell the same story. For
this exercise, treat observations as independent and unweighted. Because the complex survey design is
ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

The estimated mean shows that the adults in the sample report an average of about 7.62 hours of sleep on weekdays.

At 90% confidence, the population mean lies between 7.5841 and 7.6584 hours. At 95% confidence, the population mean lies between 7.57 and 7.66 hours. At 99% confidence, the population mean lies between 7.5631 and 7.6794 hours. 

In each of our three intervals, we find thta the difference is statistically significant as 8 hours is outside all three intervals. However, practically, we cannot say entirely that the difference of 23 minutes in our estimated sleep hours from 8 hours is significant to everyday human experience. Each person needs a different amount of sleep and we require further insight into what the researchers want to define as a difference in the number of hours in sleep to tell whether this is practically significant to the average person in the population.

We cannot say that the results represent all adults because we are to treat the observations as independent - stated in the problem that we are ignoring national resentation or the fact that the intervals were adjusted too. 